# Train EfficientNetB0 Food Classifier

This notebook trains a TensorFlow EfficientNetB0 transfer learning model for burger, pizza, and salad classification.

## 1. Imports and Settings

In [1]:
import sys
print(sys.executable)

c:\Users\biyan\Documents\python\Deep_Learning\ANN_siddhi\venv\python.exe


In [4]:
from pathlib import Path

import tensorflow as tf

PROJECT_ROOT = Path('..').resolve()

DATASET_DIR = PROJECT_ROOT / 'dataset'
MODEL_OUTPUT_DIR = PROJECT_ROOT.parent / 'backend' / 'saved_model'
MODEL_OUTPUT_PATH = MODEL_OUTPUT_DIR / 'food_effnetb0.keras'

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
SEED = 42

DATASET_DIR, MODEL_OUTPUT_PATH

(WindowsPath('C:/Users/biyan/Documents/GitHub/CNN_project/model_training/dataset'),
 WindowsPath('C:/Users/biyan/Documents/GitHub/CNN_project/backend/saved_model/food_effnetb0.keras'))

In [5]:
print(DATASET_DIR)
print(DATASET_DIR.exists())

C:\Users\biyan\Documents\GitHub\CNN_project\model_training\dataset
True


## 2. Load Train, Validation, and Test Datasets

In [6]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'train',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'val',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'test',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
class_names

Found 90 files belonging to 3 classes.
Found 30 files belonging to 3 classes.
Found 30 files belonging to 3 classes.


['burger', 'pizza', 'salad']

## 3. Improve Dataset Loading Speed

In [7]:
autotune = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(autotune)
val_ds = val_ds.prefetch(autotune)
test_ds = test_ds.prefetch(autotune)

## 4. Build EfficientNetB0 Transfer Learning Model

In [8]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ],
    name='data_augmentation',
)

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(*IMAGE_SIZE, 3),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='smart_food_effnetb0')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()



Model: "smart_food_effnetb0"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 data_augmentation (Sequent  (None, 224, 224, 3)       0         
 ial)                                                            
                                                                 
 efficientnetb0 (Functional  (None, 7, 7, 1280)        4049571   
 )                                                               
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                             

## 5. Train the Model

In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_OUTPUT_PATH),
        monitor='val_accuracy',
        save_best_only=True,
    ),
]

history = model.fit(    
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

Epoch 1/10


3/3 [==============================] - 30s 5s/step - loss: 1.1022 - accuracy: 0.4889 - val_loss: 0.9357 - val_accuracy: 0.6333
Epoch 2/10
3/3 [==============================] - 6s 2s/step - loss: 0.8472 - accuracy: 0.7111 - val_loss: 0.7175 - val_accuracy: 0.7667
Epoch 3/10
3/3 [==============================] - 6s 2s/step - loss: 0.6720 - accuracy: 0.8111 - val_loss: 0.5487 - val_accuracy: 0.8667
Epoch 4/10
3/3 [==============================] - 6s 2s/step - loss: 0.5500 - accuracy: 0.9111 - val_loss: 0.4225 - val_accuracy: 0.9667
Epoch 5/10
3/3 [==============================] - 4s 1s/step - loss: 0.4588 - accuracy: 0.9000 - val_loss: 0.3300 - val_accuracy: 0.9667
Epoch 6/10
3/3 [==============================] - 4s 1s/step - loss: 0.3486 - accuracy: 0.9556 - val_loss: 0.2649 - val_accuracy: 0.9667
Epoch 7/10
3/3 [==============================] - 4s 1s/step - loss: 0.3028 - accuracy: 0.9444 - val_loss: 0.2206 - val_accuracy: 0.9667
Epoch 8/10
3/3 [======================

## 6. Evaluate on Test Set

In [11]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

1/1 [==============================] - 1s 993ms/step - loss: 0.0989 - accuracy: 1.0000
Test loss: 0.0989
Test accuracy: 1.0000


## 7. Save Model and Class Names

In [12]:
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_OUTPUT_PATH)

labels_path = MODEL_OUTPUT_DIR / 'class_names.txt'
labels_path.write_text('\n'.join(class_names), encoding='utf-8')

print(f'Saved model to: {MODEL_OUTPUT_PATH}')
print(f'Saved class names to: {labels_path}')

Saved model to: C:\Users\biyan\Documents\GitHub\CNN_project\backend\saved_model\food_effnetb0.keras
Saved class names to: C:\Users\biyan\Documents\GitHub\CNN_project\backend\saved_model\class_names.txt
